
# ESPAC Livestock LCI -> XML Generator

Generate one XML file per row from the livestock LCI pipeline using a livestock XML template whose exchange `generalComment` values match livestock table columns.

This notebook is independent from the crop XML notebook and inherits the livestock summary strategy from notebook 2 metadata when available.

Important note:
- a dedicated livestock XML template is required at `inputs/04-05_Model_livestock.XML` unless you change the path below;
- this notebook intentionally does not reuse the crop XML model as a silent fallback.


In [5]:

from pathlib import Path
import re
import json
import xml.etree.ElementTree as ET
from datetime import date

import pandas as pd
import numpy as np
from IPython.display import display, Markdown

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'inputs').exists() and (PROJECT_DIR.parent / 'inputs').exists():
    PROJECT_DIR = PROJECT_DIR.parent

DEFAULT_SUMMARY_STRATEGY = 'national'
LATEST_FILTERED_SUMMARY_META_PATH = PROJECT_DIR / 'outputs/02_latest_livestock_filtered_export_summary.json'
summary_meta = {}
if LATEST_FILTERED_SUMMARY_META_PATH.exists():
    try:
        summary_meta = json.loads(LATEST_FILTERED_SUMMARY_META_PATH.read_text(encoding='utf-8')) or {}
    except Exception:
        summary_meta = {}

available_summary_filtered = sorted(
    (PROJECT_DIR / 'outputs/CSVs').glob('02_espac_livestock_lci_table_filtered__summary_*.csv'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

_meta_summary = str(summary_meta.get('summary_level', '')).strip()
if _meta_summary:
    SUMMARY_STRATEGY = _meta_summary
    SUMMARY_SOURCE = 'metadata'
elif available_summary_filtered:
    m_latest = re.search(r'__summary_([a-z0-9_]+)$', available_summary_filtered[0].stem)
    SUMMARY_STRATEGY = m_latest.group(1) if m_latest else DEFAULT_SUMMARY_STRATEGY
    SUMMARY_SOURCE = 'latest_filtered_csv'
else:
    SUMMARY_STRATEGY = DEFAULT_SUMMARY_STRATEGY
    SUMMARY_SOURCE = 'default'

SUMMARY_TOKEN = re.sub(r"[^a-z0-9]+", "_", str(SUMMARY_STRATEGY).strip().lower()).strip("_") or 'province'
DEFAULT_TEMPLATE = PROJECT_DIR / 'inputs/04-05_Model_livestock.XML'
AVAILABLE_XML_TEMPLATES = sorted((PROJECT_DIR / 'inputs').glob('livestock*.XML'))
DEFAULT_LCI = PROJECT_DIR / f'outputs/CSVs/03-05_espac_livestock_lci_table_filtered_dfe__summary_{SUMMARY_TOKEN}.csv'
if not DEFAULT_LCI.exists():
    DEFAULT_LCI = PROJECT_DIR / f'outputs/CSVs/02_espac_livestock_lci_table_filtered__summary_{SUMMARY_TOKEN}.csv'
DEFAULT_UNCERTAINTY = DEFAULT_LCI.with_name(f"{DEFAULT_LCI.stem}_uncertainty.csv")
XML_EXPORTS_ROOT = PROJECT_DIR / 'outputs/05_xml_exports_livestock_lci'
OUTPUT_DIR = XML_EXPORTS_ROOT / f'summary_{SUMMARY_TOKEN}'

print(f'Inherited livestock summary strategy: {SUMMARY_STRATEGY} (source: {SUMMARY_SOURCE})')
print(f'Expected livestock LCI table: {DEFAULT_LCI}')
print(f'Expected livestock uncertainty table: {DEFAULT_UNCERTAINTY}')
print('Livestock XML templates:')
for template_path in AVAILABLE_XML_TEMPLATES:
    print(' -', template_path.name)

if not AVAILABLE_XML_TEMPLATES:
    display(Markdown('No livestock XML templates found in `inputs/`. Add templates named like `livestock00001.XML`, `livestock00002.XML`, etc., then rerun this notebook.'))


Inherited livestock summary strategy: product (source: metadata)
Expected livestock LCI table: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\03-05_espac_livestock_lci_table_filtered_dfe__summary_product.csv
Expected livestock uncertainty table: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\03-05_espac_livestock_lci_table_filtered_dfe__summary_product_uncertainty.csv
Livestock XML templates:
 - livestock00001.XML
 - livestock00002.XML
 - livestock00003.XML


In [6]:

LCI_TABLE_PATH = DEFAULT_LCI
UNCERTAINTY_TABLE_PATH = DEFAULT_UNCERTAINTY

if AVAILABLE_XML_TEMPLATES and LCI_TABLE_PATH.exists():
    df = pd.read_csv(LCI_TABLE_PATH)
    uncertainty_path = UNCERTAINTY_TABLE_PATH
    if uncertainty_path.exists():
        df_unc = pd.read_csv(uncertainty_path)
        unc_cols = [c for c in df_unc.columns if isinstance(c, str) and c.endswith(('__minValue', '__maxValue'))]
        merge_keys = [c for c in ['Region', 'Province', 'Product', 'System'] if c in df.columns and c in df_unc.columns]
        if len(df_unc) == len(df) and unc_cols:
            df = pd.concat([df.reset_index(drop=True), df_unc[unc_cols].reset_index(drop=True)], axis=1)
        elif merge_keys and unc_cols:
            right = df_unc[merge_keys + unc_cols].drop_duplicates(subset=merge_keys, keep='first')
            base = df.reset_index().rename(columns={'index': '__row_id'})
            df = base.merge(right, on=merge_keys, how='left').sort_values('__row_id').drop(columns='__row_id').reset_index(drop=True)
    print(f'Rows: {len(df):,}  Columns: {len(df.columns)}')
    display(df.head(3))


Rows: 7  Columns: 450


,Product,count,Normalized_product_output_kg,Area_ha_per_1kg_product,Animals_total_live_weight_kg_per_1kg_product,Producing_animals_live_weight_kg_per_1kg_product,Animals_sold_live_weight_kg_per_1kg_product,Water_l_per_1kg_product,Electricity_kWh_per_1kg_product,Live_weight_equivalent_kg_per_1kg_product,...,kgN2O_manure_mgmt_direct_per_1kg_product__minValue,kgN2O_manure_mgmt_direct_per_1kg_product__maxValue,kgN2O_manure_mgmt_indirect_per_1kg_product__minValue,kgN2O_manure_mgmt_indirect_per_1kg_product__maxValue,kgN2O_manure_mgmt_total_per_1kg_product__minValue,kgN2O_manure_mgmt_total_per_1kg_product__maxValue,kgN2_manure_mgmt_per_1kg_product__minValue,kgN2_manure_mgmt_per_1kg_product__maxValue,kgN_manure_after_management_per_1kg_product__minValue,kgN_manure_after_management_per_1kg_product__maxValue
0,cattle_live,12441,1.0,0.002750,2.000000,0.909091,NaN,0.032593,0.000000,2.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,eggs,251,1.0,NaN,4.768288,4.768288,NaN,0.000027,0.000001,4.768288,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,goat_live,115,1.0,0.126024,2.173913,NaN,NaN,0.224480,0.000000,2.173913,...,0.002152,0.002152,0.000158,0.000158,0.00231,0.00231,0.004109,0.004109,0.578939,0.578939


In [7]:

def normalize_key(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r'[^a-z0-9_]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s


def to_numeric_or_default(v, default=0.0):
    try:
        if pd.isna(v):
            return float(default)
        return float(v)
    except Exception:
        return float(default)


def safe_name(*parts):
    txt = '_'.join([str(p).strip() for p in parts if str(p).strip() and str(p).strip().lower() != 'nan'])
    txt = re.sub(r'[^A-Za-z0-9._-]+', '_', txt).strip('_')
    return txt[:180] if txt else 'record'


def get_root_and_ns(template_xml_path: Path):
    tree = ET.parse(template_xml_path)
    root = tree.getroot()
    m = re.match(r'\{(.+)\}', root.tag)
    ns = {'spold': m.group(1)} if m else {}
    return tree, root, ns


def qname(local_name: str, ns: dict):
    if ns and ns.get('spold'):
        return f"{{{ns['spold']}}}{local_name}"
    return local_name


def parse_template_product_tokens(template_path: Path):
    tree = ET.parse(template_path)
    root = tree.getroot()
    m = re.match(r'\{(.+)\}', root.tag)
    ns = {'spold': m.group(1)} if m else {}
    for ex in root.findall('.//' + qname('exchange', ns)):
        output_group = ex.find(qname('outputGroup', ns))
        if output_group is not None and str(output_group.text or '').strip() == '0':
            comment = str(ex.attrib.get('generalComment', '')).strip()
            if comment:
                return [token.strip().lower() for token in comment.split(',') if token.strip()]
    return []


TEMPLATE_PRODUCT_MAP = {}
for template_path in AVAILABLE_XML_TEMPLATES:
    for token in parse_template_product_tokens(template_path):
        TEMPLATE_PRODUCT_MAP.setdefault(token, template_path)


def template_for_product(product: str):
    if not product:
        return None
    return TEMPLATE_PRODUCT_MAP.get(str(product).strip().lower())


def parse_comment_expression(comment: str):
    parts = [p.strip() for p in re.split(r'\s*\+\s*', str(comment).strip()) if p.strip()]
    return parts


def evaluate_comment_expression(row: pd.Series, comment: str):
    cols = parse_comment_expression(comment)
    if not cols:
        return None
    values = []
    for col in cols:
        if col not in row.index:
            return None
        values.append(to_numeric_or_default(row.get(col), default=0.0))
    return sum(values)


def evaluate_uncertainty(row: pd.Series, comment: str):
    cols = parse_comment_expression(comment)
    if not cols:
        return None
    mins = []
    maxs = []
    for col in cols:
        lo = row.get(f'{col}__minValue')
        hi = row.get(f'{col}__maxValue')
        if pd.isna(lo) or pd.isna(hi):
            return None
        mins.append(to_numeric_or_default(lo, default=0.0))
        maxs.append(to_numeric_or_default(hi, default=0.0))
    return sum(mins), sum(maxs)


def apply_row_to_xml(tree, root, ns, row: pd.Series):
    for ex in root.findall('.//' + qname('exchange', ns)):
        comment = str(ex.attrib.get('generalComment', '')).strip()
        if not comment:
            continue
        # Set exchange name for output exchange (outputGroup == '0')
        output_group = ex.find(qname('outputGroup', ns))
        if output_group is not None and str(output_group.text or '').strip() == '0':
            product_name = str(row.get('Product', '')).strip()
            system_name = str(row.get('System', '')).strip()
            if product_name:
                ex.attrib['name'] = f"{product_name}, {system_name}".rstrip(', ')
        value = evaluate_comment_expression(row, comment)
        if value is None:
            continue
        ex.attrib['meanValue'] = str(value)
        uncertainty = evaluate_uncertainty(row, comment)
        if uncertainty is not None:
            lo, hi = uncertainty
            ex.attrib['uncertaintyType'] = '3'
            ex.attrib['minValue'] = str(lo)
            ex.attrib['maxValue'] = str(hi)

    for aname in root.findall('.//' + qname('activityName', ns)):
        aname.text = safe_name(row.get('Product', ''), row.get('System', ''), row.get('Region', ''), row.get('Province', ''))
    return tree



In [8]:
# SimaPro XML Correction Constants and Functions
DEFAULT_XML_REVIEWER_NAME = "ESPAC Project"
DEFAULT_XML_VALIDATION_METADATA = {
    "reviewStatus": "draft",
    "reviewer": DEFAULT_XML_REVIEWER_NAME,
    "proofReadingValidator": {
        "name": DEFAULT_XML_REVIEWER_NAME,
        "email": "n/a",
        "companyCode": "ESPAC"
    }
}

SIMAPRO_SCHEMA_LOCATION = (
    "http://www.EcoInvent.org/EcoSpold01 "
    "http://www.EcoInvent.org/EcoSpold01..\\..\\EcoSpold01Dataset.xsd "
    "https://github.com/brightway-lca/pyecospold/tree/main/pyecospold/schemas/v1"
)

SIMAPRO_CLASSIFICATION = {
    "category": "Agricultural",
    "localCategory": "Agricultural",
    "subCategory": "ECUADOR",
    "localSubCategory": "ECUADOR"
}

XML_VALIDATION_METADATA = DEFAULT_XML_VALIDATION_METADATA.copy()


def _qname(local_name: str, ns: dict) -> str:
    if ns and ns.get("spold"):
        return f"{{{ns['spold']}}}{local_name}"
    return local_name


def _find_or_create(parent, child_name: str, ns: dict):
    child = parent.find(f"spold:{child_name}", ns) if ns else parent.find(child_name)
    if child is None:
        child = ET.SubElement(parent, _qname(child_name, ns))
    return child


def _reorder_modelling_and_validation_children(modelling, representativeness, validation, source):
    children = list(modelling)
    ordered = []
    for node in (representativeness, validation, source):
        if node is not None and node in children and node not in ordered:
            ordered.append(node)
    for node in children:
        if node not in ordered:
            ordered.append(node)
    for node in list(modelling):
        modelling.remove(node)
    for node in ordered:
        modelling.append(node)


def _prepare_tree_for_simapro_serialization(tree, ns: dict):
    if ns and ns.get("spold"):
        ET.register_namespace("", ns["spold"])
    ET.register_namespace("xsi", "http://www.w3.org/2001/XMLSchema-instance")
    root = tree.getroot()
    root.attrib["{http://www.w3.org/2001/XMLSchema-instance}schemaLocation"] = SIMAPRO_SCHEMA_LOCATION


def _ensure_simapro_validation_metadata(root, ns: dict, validation_meta: dict, generation_date: str):
    meta = root.find('.//spold:metaInformation', ns) if ns else root.find('.//metaInformation')
    if meta is None:
        return
    modelling = _find_or_create(meta, 'modellingAndValidation', ns)
    representativeness = _find_or_create(modelling, 'representativeness', ns)
    source = _find_or_create(modelling, 'source', ns)
    validation = _find_or_create(modelling, 'validation', ns)

    def _set_text(parent, child_name, value):
        child = _find_or_create(parent, child_name, ns)
        child.text = "" if value is None else str(value)

    _set_text(validation, 'reviewStatus', validation_meta.get('reviewStatus', 'draft'))
    _set_text(validation, 'reviewer', validation_meta.get('reviewer', ''))
    _set_text(validation, 'reviewDate', generation_date)

    validator_meta = validation_meta.get('proofReadingValidator', {}) or {}
    proof = _find_or_create(validation, 'proofReadingValidator', ns)
    _set_text(proof, 'name', validator_meta.get('name', ''))
    _set_text(proof, 'email', validator_meta.get('email', ''))
    _set_text(proof, 'companyCode', validator_meta.get('companyCode', ''))
    _set_text(proof, 'proofReadingDate', generation_date)

    _reorder_modelling_and_validation_children(modelling, representativeness, validation, source)


def _apply_exchange_location_unit_rules(ex, geography_location: str):
    # SimaPro requires exchange #1 location to match dataset geography.
    number_raw = str(ex.attrib.get("number", "")).strip()
    try:
        number = int(number_raw)
    except Exception:
        number = None

    if number == 1:
        ex.attrib["location"] = geography_location or ex.attrib.get("location", "GLO")
    else:
        ex.attrib["location"] = "RER"

    if number is not None and 19 <= number <= 25:
        ex.attrib["unit"] = "kg"


def _ensure_exchange_group_tag(ex, ns: dict):
    def _lname(tag):
        return tag.split("}", 1)[1] if isinstance(tag, str) and tag.startswith("{") and "}" in tag else str(tag)

    has_group = any(_lname(child.tag) in {"inputGroup", "outputGroup"} for child in list(ex))
    if has_group:
        return

    # Elementary/resource flows use outputGroup=4; technosphere defaults to inputGroup=5.
    if "category" in ex.attrib:
        group = ET.SubElement(ex, _qname("outputGroup", ns))
        group.text = "4"
    else:
        group = ET.SubElement(ex, _qname("inputGroup", ns))
        group.text = "5"


def get_exchange_elements(root, ns: dict):
    if ns:
        return root.findall('.//spold:flowData/spold:exchange', ns)
    return root.findall('.//flowData/exchange')

In [9]:
def render_xml_for_livestock_row(template_xml_path: Path, row: pd.Series, generation_date=None):
    """
    Render corrected XML for a livestock row with SimaPro compatibility.
    Applies namespace registration, validation metadata, location/unit rules, and exchange groups.
    """
    tree, root, ns = get_root_and_ns(template_xml_path)
    generation_date = generation_date or date.today().isoformat()
    
    # Apply SimaPro validation metadata
    _ensure_simapro_validation_metadata(root, ns, XML_VALIDATION_METADATA, generation_date)
    
    # Get geography location for reference exchange
    if ns:
        geo = root.find('.//spold:geography', ns)
    else:
        geo = root.find('.//geography')
    geography_location = geo.attrib.get("location", "GLO") if geo is not None else "GLO"
    
    # Apply corrections to each exchange
    for ex in get_exchange_elements(root, ns):
        # Apply location and unit rules
        _apply_exchange_location_unit_rules(ex, geography_location)
        # Ensure inputGroup/outputGroup tags exist
        _ensure_exchange_group_tag(ex, ns)
        
        comment = str(ex.attrib.get("generalComment", "")).strip()
        if not comment:
            continue
        
        # Set exchange name for output exchange (outputGroup == '0')
        output_group = ex.find(_qname('outputGroup', ns))
        if output_group is not None and str(output_group.text or "").strip() == "0":
            product_name = str(row.get('Product', '')).strip()
            system_name = str(row.get('System', '')).strip()
            if product_name:
                ex.attrib['name'] = f"{product_name}, {system_name}".rstrip(', ')
        
        # Evaluate exchange value from row
        value = evaluate_comment_expression(row, comment)
        if value is not None:
            ex.attrib['meanValue'] = str(value)
        
        # Evaluate and set uncertainty bounds
        uncertainty = evaluate_uncertainty(row, comment)
        if uncertainty is not None:
            lo, hi = uncertainty
            ex.attrib['uncertaintyType'] = '3'
            ex.attrib['minValue'] = str(lo)
            ex.attrib['maxValue'] = str(hi)
        else:
            # For non-reference exchanges, still set uncertainty type
            if value is not None:
                number_raw = str(ex.attrib.get("number", "")).strip()
                try:
                    number = int(number_raw)
                    if number != 1:  # Don't apply uncertainty to reference exchange
                        ex.attrib["uncertaintyType"] = "3"
                        ex.attrib["minValue"] = str(value)
                        ex.attrib["maxValue"] = str(value)
                except Exception:
                    pass
    
    # Set reference product and process names
    if ns:
        ref = root.find('.//spold:referenceFunction', ns)
        ex1 = root.find('.//spold:flowData/spold:exchange[@number="1"]', ns)
    else:
        ref = root.find('.//referenceFunction')
        ex1 = root.find('.//flowData/exchange[@number="1"]')
    
    proc_name = safe_name(row.get('Product', ''), row.get('System', ''), row.get('Region', ''), row.get('Province', ''))
    
    if ref is not None:
        ref.attrib['name'] = proc_name
        ref.attrib['localName'] = proc_name
        ref.attrib['category'] = SIMAPRO_CLASSIFICATION.get("category", "Agricultural")
        ref.attrib['localCategory'] = SIMAPRO_CLASSIFICATION.get("localCategory", "Agricultural")
        ref.attrib['subCategory'] = SIMAPRO_CLASSIFICATION.get("subCategory", "ECUADOR")
        ref.attrib['localSubCategory'] = SIMAPRO_CLASSIFICATION.get("localSubCategory", "ECUADOR")
    
    if ex1 is not None:
        ex1.attrib['name'] = proc_name
        # SimaPro compatibility: keep reference product exchange free of uncertainty bounds
        for attr in ('uncertaintyType', 'minValue', 'maxValue'):
            ex1.attrib.pop(attr, None)
    
    # Update activity name
    for aname in root.findall('.//' + _qname('activityName', ns)):
        aname.text = proc_name
    
    # Prepare tree for serialization (namespace registration and schemaLocation)
    _prepare_tree_for_simapro_serialization(tree, ns)
    
    return tree

In [10]:

written = []
if AVAILABLE_XML_TEMPLATES and LCI_TABLE_PATH.exists():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    generation_date = date.today().isoformat()
    for idx, row in df.iterrows():
        product = str(row.get('Product', '')).strip()
        if not product or '(unknown)' in product.lower():
            continue  # Skip rows with empty or unknown product
        # Skip rows with too many empty columns (malformed data)
        non_empty_count = sum(1 for v in row.values if pd.notna(v) and str(v).strip())
        if non_empty_count < 20:  # Adjust threshold as needed
            continue
        system = str(row.get('System', '')).strip().replace('(unknown)', '').strip()
        region = str(row.get('Region', '')).strip().replace('(unknown)', '').strip()
        province = str(row.get('Province', '')).strip().replace('(unknown)', '').strip()
        template_path = template_for_product(product)
        if template_path is None:
            if DEFAULT_TEMPLATE.exists():
                template_path = DEFAULT_TEMPLATE
            else:
                template_path = AVAILABLE_XML_TEMPLATES[0]
        # Use corrected rendering function with SimaPro compatibility
        tree = render_xml_for_livestock_row(template_path, row, generation_date=generation_date)
        filename = safe_name(product, system, region, province, SUMMARY_TOKEN) + '.xml'
        out_path = OUTPUT_DIR / filename
        tree.write(out_path, encoding='UTF-8', xml_declaration=True)
        written.append(out_path)
    print(f'Wrote {len(written):,} livestock XML files to {OUTPUT_DIR}')
    if written:
        display(Markdown(f'Sample output: `{written[0]}`'))


Wrote 7 livestock XML files to c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\05_xml_exports_livestock_lci\summary_product


Sample output: `c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\05_xml_exports_livestock_lci\summary_product\cattle_live_product.xml`